# Hotel Cancellation — Final Model


In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

In [2]:
df = pd.read_csv("hotel_bookings.csv")
df["has_agent"] = df["agent"].notna().astype(int)
df["has_company"] = df["company"].notna().astype(int)
df = df.drop(columns=["agent", "company"])
df = df.dropna(subset=["country", "children"]).copy()
df = df[~((df["adults"] == 0) & (df["children"] == 0) & (df["babies"] == 0))].copy()
df = df[(df["adr"] > 0) & (df["adr"] < 5400)].copy()
df = df[~((df["adults"] > 10) | (df["children"] > 6) | (df["babies"] > 4) | (df["required_car_parking_spaces"] > 3))].copy()
df["total_stay_nights"] = df["stays_in_week_nights"] + df["stays_in_weekend_nights"]

X = df.drop(columns=["is_canceled", "reservation_status", "reservation_status_date", "assigned_room_type", "booking_changes", "days_in_waiting_list"])
y = df["is_canceled"]
print("Dataset:", X.shape)

Dataset: (116922, 27)


In [3]:
groups = pd.factorize(pd.util.hash_pandas_object(X, index=False))[0]
train_groups, test_groups = train_test_split(pd.unique(groups), test_size=0.2, random_state=42)
train_mask = pd.Series(groups, index=X.index).isin(train_groups)
X_train, X_test = X.loc[train_mask], X.loc[~train_mask]
y_train, y_test = y.loc[train_mask], y.loc[~train_mask]

In [4]:
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(include="object").columns.tolist()
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.1941574456226751,
        max_iter=108,
        max_leaf_nodes=40,
        l2_regularization=1.8977710745066665,
        random_state=42
    ))
])
best_model.fit(X_train, y_train)
print("Final model trained")

Final model trained


In [5]:
model_package = {
    "model": best_model,
    "threshold": 0.36,
    "features": X_train.columns.tolist()
}
joblib.dump(model_package, "hotel_cancellation_model.joblib")
print("Saved: hotel_cancellation_model.joblib")
print("Features:", len(model_package["features"]))
print("Threshold:", model_package["threshold"])

Saved: hotel_cancellation_model.joblib
Features: 27
Threshold: 0.36


In [6]:

import json

sample_booking = X_test.iloc[0].to_dict()

print(json.dumps(sample_booking, indent=2, default=str))


{
  "hotel": "Resort Hotel",
  "lead_time": 77,
  "arrival_date_year": 2015,
  "arrival_date_month": "July",
  "arrival_date_week_number": 27,
  "arrival_date_day_of_month": 1,
  "stays_in_weekend_nights": 2,
  "stays_in_week_nights": 5,
  "adults": 2,
  "children": 0.0,
  "babies": 0,
  "meal": "BB",
  "country": "PRT",
  "market_segment": "Online TA",
  "distribution_channel": "TA/TO",
  "is_repeated_guest": 0,
  "previous_cancellations": 0,
  "previous_bookings_not_canceled": 0,
  "reserved_room_type": "A",
  "deposit_type": "No Deposit",
  "customer_type": "Transient",
  "adr": 94.0,
  "required_car_parking_spaces": 0,
  "total_of_special_requests": 0,
  "has_agent": 1,
  "has_company": 0,
  "total_stay_nights": 7
}
